# 02 · Generación de datos sintéticos — TodoComponentes

En este notebook se:
1. Autentica contra BigQuery (mismo patrón que el notebook 01).
2. Genera datos sintéticos con **Faker** para las 7 tablas, respetando:
   - Volúmenes mínimos: ~500 clientes, 70 productos, ~2000 pedidos, ~4500 líneas, 1 pago/pedido, ~35% de reseñas.
   - Coherencia de negocio: `order_date < shipped_date < delivered_date`, `unit_price` de la línea como *snapshot*, estados coherentes entre pedido → pago → reseña.
3. Carga todo a BigQuery vía DataFrames (`load_table_from_dataframe`).
4. Valida conteos y consistencias.

> **Nota:** el dataset ya existe por el notebook [`01_setup_bigquery.ipynb`](./01_setup_bigquery.ipynb). Si se quiere regenerar de cero, las cargas usan `WRITE_TRUNCATE` por tabla (idempotente).

In [30]:
# --- 1. Autenticación y cliente BigQuery ---
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account


def _find_project_root() -> Path:
    for cand in (Path.cwd(), *Path.cwd().parents):
        if (cand / ".env").exists() or (cand / ".env.example").exists():
            return cand
    return Path.cwd().parents[2]


PROJECT_ROOT = _find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
DATASET_ID = os.environ["BQ_DATASET_ID"]
CREDENTIALS_PATH = str((PROJECT_ROOT / os.environ["GOOGLE_APPLICATION_CREDENTIALS"]).resolve())
assert os.path.exists(CREDENTIALS_PATH), f"Credenciales no encontradas en: {CREDENTIALS_PATH}"

credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH, scopes=["https://www.googleapis.com/auth/bigquery",
                              "https://www.googleapis.com/auth/bigquery.readonly"]
)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials, location="EU")

# Descubrir la región real del dataset para evitar 404
dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
_ds = client.get_dataset(dataset_id)
LOCATION = _ds.location
print(f"Proyecto : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID} (región={LOCATION})")

# Parámetros de generación (volúmenes mínimos del enunciado)
SEED = 42
N_CATEGORIES   = 6
N_CUSTOMERS    = 500
N_PRODUCTS     = 70
N_ORDERS       = 2000
REVIEW_RATE    = 0.35   # ~35% de líneas con reseña

Proyecto : todocomponentes
Dataset  : todo_componentes (región=EU)


In [31]:
# Países/ciudades de envío frecuentes (mercado europeo principal + otros)
# Cada entrada: (país, ciudad, locale Faker)
SHIPPING = [
    ("España", "Madrid",        "es_ES"),
    ("España", "Barcelona",     "es_ES"),
    ("España", "Valencia",      "es_ES"),
    ("España", "Sevilla",       "es_ES"),
    ("España", "Bilbao",        "es_ES"),
    ("Francia", "París",        "fr_FR"),
    ("Francia", "Lyon",         "fr_FR"),
    ("Alemania", "Berlín",      "de_DE"),
    ("Alemania", "Múnich",      "de_DE"),
    ("Italia", "Milán",         "it_IT"),
    ("Portugal", "Lisboa",      "pt_PT"),
    ("México", "CDMX",          "es_MX"),
    ("Colombia", "Bogotá",      "es_CO"),
    ("Argentina", "Buenos Aires","es_AR"),
]

# Locale Faker por país (para teléfonos realistas)
LOCALE_BY_COUNTRY = {
    "España": "es_ES", "Francia": "fr_FR", "Alemania": "de_DE",
    "Italia": "it_IT", "Portugal": "pt_PT",
    "México": "es_MX", "Colombia": "es_CO", "Argentina": "es_AR",
}

In [32]:
# --- 2. Utilidades, catálogos base y helpers ---
import random
from datetime import datetime, timedelta
from decimal import Decimal
from faker import Faker

random.seed(SEED)
fake = Faker("es_ES")   # instancia global (descripciones, comentarios)

def money(x) -> Decimal:
    """Redondea a 2 decimales (importes)."""
    return Decimal(str(x)).quantize(Decimal("0.01"))

def past_datetime(max_days_ago: int, min_days_ago: int = 0) -> datetime:
    """Fecha/hora aleatoria entre (hoy - max_days) y (hoy - min_days), sin microsegundos."""
    d = datetime.now() - timedelta(days=random.randint(min_days_ago, max_days_ago))
    return d.replace(hour=random.randint(0, 23), minute=random.randint(0, 59),
                     second=random.randint(0, 59), microsecond=0)

# --- Catálogos base ---
CHANNELS = ["organic_search", "paid_search", "email", "social", "referral", "direct"]

BRANDS = ["CircuitPro", "MakersHub", "VoltParts", "TechKit", "SensorWorld",
          "ArduinoStore", "ElectroLab", "MakerShop"]

# 6 categorías (coinciden con PRODUCT_NAME_HINTS de la sección 4)
CATEGORY_ROWS = [
    (1, "Microcontroladores", "Placas de desarrollo y MCUs para prototipos"),
    (2, "Sensores",           "Sensores ambientales, de movimiento y de fuerza"),
    (3, "Display y HMI",      "Pantallas, matrices LED e interfaces táctiles"),
    (4, "Almacenamiento",     "Tarjetas SD, módulos eMMC, flash y SSD"),
    (5, "Energía y fuentes",  "Baterías, cargadores y reguladores de tensión"),
    (6, "Conectividad",       "Módulos WiFi, BLE, LoRa y antenas"),
]

In [33]:
# --- 3. Categorías y clientes ---
df_categories = pd.DataFrame(CATEGORY_ROWS, columns=["category_id", "name", "description"])

rows = []
for i in range(1, N_CUSTOMERS + 1):
    country, city, locale = random.choice(SHIPPING)
    f_local = Faker(locale)
    first, last = f_local.first_name(), f_local.last_name()
    rows.append({
        "customer_id": i,
        "first_name": first,
        "last_name": last,
        "email": f_local.email(),
        "phone": f_local.phone_number(),
        "country": country,
        "city": city,
        "acquisition_channel": random.choice(CHANNELS),
        "created_at": past_datetime(400, 30),
    })
df_customers = pd.DataFrame(rows)

print(df_categories)
print()
print(df_customers.head())

   category_id                name  \
0            1  Microcontroladores   
1            2            Sensores   
2            3       Display y HMI   
3            4      Almacenamiento   
4            5   Energía y fuentes   
5            6        Conectividad   

                                       description  
0      Placas de desarrollo y MCUs para prototipos  
1  Sensores ambientales, de movimiento y de fuerza  
2    Pantallas, matrices LED e interfaces táctiles  
3           Tarjetas SD, módulos eMMC, flash y SSD  
4    Baterías, cargadores y reguladores de tensión  
5                Módulos WiFi, BLE, LoRa y antenas  

   customer_id first_name last_name                      email  \
0            1   Lisandro   Moreira      araujoema@example.com   
1            2     Gracia     Lladó          salba@example.org   
2            3   Hartmuth  Lachmann     eberhard22@example.org   
3            4     Felisa   Barrios  camilafajardo@example.org   
4            5    Pascale   Mei

In [34]:
# --- 4. Productos (70) enlazados a categorías ---
PRODUCT_NAME_HINTS = {
    "Microcontroladores":    ["Arduino Nano", "Arduino Uno R3", "ESP32 DevKitC", "ESP32-S3 WROOM",
                              "STM32F4 Discovery", "Raspberry Pi Pico", "Teensy 4.1", "ATSAMD21 XPRO"],
    "Sensores":              ["Sensor DHT22", "Sensor BME280", "IMU MPU-6050", "Sensor MQ-135",
                              "Modulo PIR HC-SR501", "Sensor de fuerza FSR 401", "LDR 5mm", "Encoder AS5048P"],
    "Display y HMI":         ["LCD 16x2 I2C", "OLED 0.96'' I2C", "Pantalla TFT 2.8''", "E-ink 2.13''",
                              "Matriz LED 8x8 MAX7219", "Tactil capacitive 4x4"],
    "Almacenamiento":        ["Tarjeta microSD 32GB A2", "Modulo eMMC 8GB", "SSD SATA 120GB",
                              "Modulo SD SPI", "Flash W25Q128 16MB"],
    "Energía y fuentes":     ["Bateria LiPo 3.7V 2200mAh", "Modulo TP4056", "Regulador AMS1117-3.3",
                              "Buck 3A 12V-5V", "Suelo de 9V", "Bateria 18650 3000mAh"],
    "Conectividad":          ["Modulo WiFi ESP8266", "Bluetooth HC-05", "Modulo LoRa SX1276 868MHz",
                              "Antena 2.4GHz IPEX", "Modulo BLE nRF52840"],
}

rows = []
cat_ids = df_categories["category_id"].tolist()
cat_names = dict(zip(df_categories["category_id"], df_categories["name"]))
for pid in range(1, N_PRODUCTS + 1):
    cat_id = random.choice(cat_ids)
    base = random.choice(PRODUCT_NAME_HINTS[cat_names[cat_id]])
    brand = random.choice(BRANDS)
    name = f"{brand} {base} v{random.randint(1,3)}"
    cost = money(random.uniform(0.8, 60.0))
    # margen 25%-120% sobre coste
    price = money(cost * Decimal(str(random.uniform(1.25, 2.2))))
    rows.append({
        "product_id": pid,
        "category_id": cat_id,
        "name": name,
        "description": fake.text(max_nb_chars=140),
        "brand": brand,
        "unit_cost": cost,
        "unit_price": price,
        "stock": random.randint(0, 500),
        "is_active": random.random() > 0.15,   # ~15% inactivos
        "created_at": past_datetime(500, 10),
    })
df_products = pd.DataFrame(rows)
df_products.head()

,product_id,category_id,name,description,brand,unit_cost,unit_price,stock,is_active,created_at
0,1,3,SensorWorld Pantalla TFT 2.8'' v3,Cómo modo media. Instituciones fútbol miedo to...,SensorWorld,4.30,8.52,333,True,2025-11-26 23:46:54
1,2,1,ElectroLab ESP32 DevKitC v1,Superior sociedad una apoyo punto abril padres...,ElectroLab,56.09,72.05,428,True,2025-12-16 05:59:47
2,3,3,CircuitPro LCD 16x2 I2C v2,Medidas como cama experiencia. Victoria santa ...,CircuitPro,34.42,46.53,171,True,2025-10-02 17:33:31
3,4,2,SensorWorld Encoder AS5048P v1,Destino incluso comercio le zonas realizar. Fu...,SensorWorld,48.68,76.14,374,True,2026-04-17 22:11:00
4,5,6,SensorWorld Modulo LoRa SX1276 868MHz v3,Decir nadie como los mes abril oficial. Materi...,SensorWorld,40.72,58.37,312,True,2025-05-24 12:52:27


In [35]:
# --- 5. Pedidos (2000) con estados coherentes y fechas ordenadas ---
ORDER_STATUSES = ["delivered", "shipped", "pending", "cancelled"]
ORDER_WEIGHTS  = [0.70,      0.12,      0.13,       0.05]

# Índices de clientes (reutilizar clientes frecuentes: reparto no uniforme)
cust_weights = [1.0 / (i + 2) for i in range(N_CUSTOMERS)]

rows = []
for oid in range(1, N_ORDERS + 1):
    cust_id = random.choices(range(1, N_CUSTOMERS + 1), weights=cust_weights, k=1)[0]
    country, city, locale = random.choice(SHIPPING)
    status = random.choices(ORDER_STATUSES, weights=ORDER_WEIGHTS, k=1)[0]

    order_date = past_datetime(365, 2)
    shipped = delivered = None
    if status in ("shipped", "delivered"):
        shipped = order_date + timedelta(days=random.randint(1, 4), hours=random.randint(0, 12))
    if status == "delivered":
        delivered = shipped + timedelta(days=random.randint(2, 9), hours=random.randint(0, 20))

    # Dirección de envío en el idioma local (más realista)
    f_local = Faker(locale)
    address = f_local.street_address()

    rows.append({
        "order_id": oid,
        "customer_id": cust_id,
        "order_status": status,
        "shipping_country": country,
        "shipping_city": city,
        "shipping_address": f"{address}, {city}, {country}",
        "order_date": order_date,
        "shipped_date": shipped,
        "delivered_date": delivered,
    })

df_orders = pd.DataFrame(rows)
print(df_orders["order_status"].value_counts(normalize=True).round(3))
print()
df_orders.head(3)

order_status
delivered    0.691
pending      0.126
shipped      0.125
cancelled    0.058
Name: proportion, dtype: float64



,order_id,customer_id,order_status,shipping_country,shipping_city,shipping_address,order_date,shipped_date,delivered_date
0,1,1,shipped,España,Madrid,"Rambla Nieves Batalla 58, Madrid, España",2026-05-14 09:02:17,2026-05-16 17:02:17,NaT
1,2,7,delivered,Colombia,Bogotá,"Diagonal 8ª # 64W-4, Bogotá, Colombia",2025-12-24 17:32:07,2025-12-25 21:32:07,2026-01-02 14:32:07
2,3,178,shipped,España,Madrid,"Camino Natalio Leal 659 Puerta 8 , Madrid, España",2026-01-21 17:13:27,2026-01-23 04:13:27,NaT


In [36]:
# --- 6. Líneas de pedido (~4500) con snapshot de precio y descuento ---
prod_lookup = df_products.set_index("product_id", drop=False)

rows = []
item_id = 0
for _, order in df_orders.iterrows():
    if order["order_status"] == "cancelled":
        n_lines = random.randint(1, 2)
    else:
        n_lines = random.choices([1, 2, 3, 4], weights=[0.10, 0.45, 0.35, 0.10], k=1)[0]

    for _ in range(n_lines):
        item_id += 1
        prod = prod_lookup.loc[random.choice(prod_lookup.index)]

        print(prod)

        qty = random.choices([1, 2, 3, 5, 10], weights=[0.50, 0.25, 0.15, 0.07, 0.03], k=1)[0]
        # snapshot: el precio en la línea puede variar +/-5% respecto al catálogo
        snapshot = money(Decimal(str(prod["unit_price"])) * Decimal(str(random.uniform(0.95, 1.05))))
        # descuento: 70% sin descuento, resto entre 5% y 25%
        discount_pct = 0 if random.random() < 0.70 else random.uniform(0.05, 0.25)
        subtotal = money(snapshot * qty)
        discount = money(subtotal * Decimal(str(discount_pct)))
        line_total = money(subtotal - discount)

        rows.append({
            "order_item_id": item_id,
            "order_id": int(order["order_id"]),
            "product_id": int(prod["product_id"]),
            "quantity": qty,
            "unit_price": snapshot,
            "discount": discount,
            "line_total": line_total,
        })

df_order_items = pd.DataFrame(rows)
print(f"Líneas generadas: {len(df_order_items)}")
print(f"Media de líneas/pedido: {len(df_order_items) / N_ORDERS:.2f}")
print()
df_order_items.head()

product_id                                                    39
category_id                                                    1
name                                    ElectroLab Teensy 4.1 v2
description    Número ex grandes carácter. Plan año aparece p...
brand                                                 ElectroLab
unit_cost                                                  33.52
unit_price                                                 43.69
stock                                                          4
is_active                                                   True
created_at                                   2026-07-15 15:54:27
Name: 39, dtype: object
product_id                                                    28
category_id                                                    6
name                            MakerShop Modulo WiFi ESP8266 v1
description    Propia mayoría derechos hijos. Centro dice est...
brand                                                  MakerShop
u

,order_item_id,order_id,product_id,quantity,unit_price,discount,line_total
0,1,1,39,1,44.01,8.97,35.04
1,2,1,28,3,80.77,0.00,242.31
2,3,2,22,5,9.20,0.00,46.00
3,4,3,45,2,27.19,0.00,54.38
4,5,3,4,2,73.75,32.63,114.87


In [37]:
# --- 7. Pagos (1 por pedido, coherentes con estado del pedido) ---
PAYMENT_METHODS = ["credit_card", "debit_card", "paypal", "bank_transfer", "bizum"]

# Mapeo de estado de pedido → distribución de estado de pago
def payment_status_for(order_status: str) -> str:
    if order_status == "cancelled":
        return random.choices(["failed", "refunded"], weights=[0.6, 0.4], k=1)[0]
    if order_status == "pending":
        return random.choices(["pending", "failed"], weights=[0.85, 0.15], k=1)[0]
    # shipped / delivered
    return random.choices(["paid", "refunded"], weights=[0.94, 0.06], k=1)[0]


rows = []
# Totales por pedido (suma de line_total)
totals_by_order = df_order_items.groupby("order_id")["line_total"].sum().to_dict()

for i, order in df_orders.iterrows():
    oid = int(order["order_id"])
    p_status = payment_status_for(order["order_status"])
    amount = totals_by_order[oid]
    # Si el pedido fue cancelado, importe pagado = 0 (se revirtió)
    if order["order_status"] == "cancelled":
        amount = Decimal("0.00")
    # paid_at: si el pago es "paid" o "refunded" → entre order_date y order_date+1h
    #          si es "pending"/"failed" → paid_at es None
    paid_at = order["order_date"] + timedelta(minutes=random.randint(0, 60)) if p_status in ("paid", "refunded") else None
    rows.append({
        "payment_id": i + 1,
        "order_id": oid,
        "payment_method": random.choice(PAYMENT_METHODS),
        "payment_status": p_status,
        "amount": amount,
        "paid_at": paid_at,
    })

df_payments = pd.DataFrame(rows)
print(df_payments["payment_status"].value_counts(normalize=True).round(3))
print()
df_payments.head(3)

payment_status
paid        0.776
pending     0.102
refunded    0.062
failed      0.059
Name: proportion, dtype: float64



,payment_id,order_id,payment_method,payment_status,amount,paid_at
0,1,1,credit_card,paid,277.35,2026-05-14 09:33:17
1,2,2,bank_transfer,paid,46.00,2025-12-24 17:33:07
2,3,3,credit_card,paid,266.22,2026-01-21 17:36:27


In [38]:
# Elegir ~35% de líneas al azar (determinista gracias a la semilla)
mask = [random.random() < REVIEW_RATE for _ in range(len(df_order_items))]
df_to_review = df_order_items[mask].copy()

In [39]:
# --- 8. Reseñas (~35% de líneas): rating, comentario y fecha coherente ---
# Cliente de cada línea: el cliente del pedido al que pertenece la línea
cust_by_order = df_orders.set_index("order_id")["customer_id"].to_dict()

COMMENTS_GOOD = [
    "Funciona a la primera, muy buen producto.",
    "Llegó antes de lo esperado, calidad impecable.",
    "Justo lo que buscaba para mi proyecto, recomendado.",
    "Compatibilidad perfecta con el ESP32.",
    "Relación calidad-precio excelente.",
    "Buen material y documentación clara en el paquete.",
]
COMMENTS_NEUTRAL = [
    "Correcto, cumple lo que promete.",
    "Bien, aunque la documentación podría mejorar.",
    "Aceptable para el precio, nada del otro mundo.",
    "Llega en buen estado, uso aún por confirmar.",
]
COMMENTS_BAD = [
    "No era lo que esperaba, queda justo para el uso previsto.",
    "La entrega se retrasó más de lo esperado.",
    "Mejorable: el cable es algo corto.",
    "Funciona, pero la calidad del acabado es justa.",
]

def rating_and_comment() -> tuple[int, str]:
    r = random.choices([1, 2, 3, 4, 5], weights=[0.03, 0.07, 0.15, 0.35, 0.40], k=1)[0]
    if r >= 4:
        return r, random.choice(COMMENTS_GOOD)
    if r == 3:
        return r, random.choice(COMMENTS_NEUTRAL)
    return r, random.choice(COMMENTS_BAD)

rows = []
for rid, (_, item) in enumerate(df_to_review.iterrows(), start=1):
    order_id = int(item["order_id"])

    # Validación: ¿existe el order_id en el diccionario?
    if order_id not in cust_by_order:
        continue

    # Validación: ¿existe el order_id en df_orders?
    order_rows = df_orders[df_orders["order_id"] == order_id]
    if order_rows.empty:
        continue

    o_date = order_rows["order_date"].iat[0]

    base = o_date + timedelta(days=random.randint(1, 30))
    r, comment = rating_and_comment()

    rows.append({
        "review_id": rid,
        "order_item_id": int(item["order_item_id"]),
        "customer_id": int(cust_by_order[order_id]),
        "rating": r,
        "comment": comment,
        "created_at": base,
    })


df_reviews = pd.DataFrame(rows)
print(f"Reseñas generadas: {len(df_reviews)}")
print(df_reviews["rating"].value_counts().sort_index().to_string())
print()
df_reviews.head()

Reseñas generadas: 1714
rating
1     43
2    115
3    264
4    609
5    683



,review_id,order_item_id,customer_id,rating,comment,created_at
0,1,1,1,4,"Llegó antes de lo esperado, calidad impecable.",2026-05-28 09:02:17
1,2,2,1,4,Relación calidad-precio excelente.,2026-05-15 09:02:17
2,3,3,7,4,"Llegó antes de lo esperado, calidad impecable.",2026-01-14 17:32:07
3,4,5,178,4,Compatibilidad perfecta con el ESP32.,2026-01-26 17:13:27
4,5,7,186,3,"Aceptable para el precio, nada del otro mundo.",2026-08-31 14:16:22


In [40]:
# --- 9. Carga a BigQuery (idempotente: WRITE_TRUNCATE por tabla) ---
import time
from google.cloud import bigquery

LOAD_ORDER = [
    ("categories",   df_categories),
    ("customers",    df_customers),
    ("products",     df_products),
    ("orders",       df_orders),
    ("order_items",  df_order_items),
    ("payments",     df_payments),
    ("reviews",      df_reviews),
]

load_job_info = []

for table_name, df in LOAD_ORDER:
    t0 = time.time()

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    job = client.load_table_from_dataframe(
        dataframe=df,
        destination=f"{PROJECT_ID}.{DATASET_ID}.{table_name}",
        job_config=job_config
    )

    job.result()  # Espera a que termine

    load_job_info.append((table_name, time.time() - t0))

print(f"\n✅ Carga completada. Total filas: {sum(r[1] for r in load_job_info)}")

c:\Users\ak471\Documents\Github\Team_Challenge_SQL\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
c:\Users\ak471\Documents\Github\Team_Challenge_SQL\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
c:\Users\ak471\Documents\Github\Team_Challenge_SQL\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
c:\Users\ak47


✅ Carga completada. Total filas: 17.75894570350647


In [46]:
# --- 10. Validación: conteos en BigQuery + integridad referencial ---
Q = f"{PROJECT_ID}.{DATASET_ID}"

print("=" * 60)
print("Conteos reales en BigQuery:")
print("=" * 60)

expected = {
    "categories":  N_CATEGORIES,
    "customers":   N_CUSTOMERS,
    "products":    N_PRODUCTS,
    "orders":      N_ORDERS,
    "order_items": len(df_order_items),
    "payments":    len(df_payments),
    "reviews":     len(df_reviews),
}

print("Q =", repr(Q))
print("names =", list(expected.keys()))

for name, exp in expected.items():
    query = f"SELECT COUNT(*) AS n FROM `{Q}.{name}`"
    job = client.query(query)
    got = list(job.result())[0].n
    ok = "✓" if got == exp else "✗"
    print(f"  {ok} {name:15s}  {got:>6d}  (esperado: {exp})")

# --- Integridad referencial (deben devolver 0 filas) ---
fk_checks = {
    "products sin categoría":    f"SELECT COUNT(*) FROM {Q}.products p LEFT JOIN {Q}.categories c ON p.category_id=c.category_id WHERE c.category_id IS NULL",
    "orders sin cliente":        f"SELECT COUNT(*) FROM {Q}.orders o LEFT JOIN {Q}.customers c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL",
    "order_items sin pedido":    f"SELECT COUNT(*) FROM {Q}.order_items oi LEFT JOIN {Q}.orders o ON oi.order_id=o.order_id WHERE o.order_id IS NULL",
    "order_items sin producto":  f"SELECT COUNT(*) FROM {Q}.order_items oi LEFT JOIN {Q}.products p ON oi.product_id=p.product_id WHERE p.product_id IS NULL",
    "payments sin pedido":       f"SELECT COUNT(*) FROM {Q}.payments py LEFT JOIN {Q}.orders o ON py.order_id=o.order_id WHERE o.order_id IS NULL",
    "reviews sin order_item":    f"SELECT COUNT(*) FROM {Q}.reviews r LEFT JOIN {Q}.order_items oi ON r.order_item_id=oi.order_item_id WHERE oi.order_item_id IS NULL",
    "reviews sin cliente":       f"SELECT COUNT(*) FROM {Q}.reviews r LEFT JOIN {Q}.customers c ON r.customer_id=c.customer_id WHERE c.customer_id IS NULL",
}

print("\nIntegridad referencial (deben ser 0):")
for label, sql in fk_checks.items():
    n = client.query(sql).result().total_rows
    mark = "✓" if n == 0 else "✗"
    print(f"  {mark} {label:30s}  {n}")

# --- Coherencia de fechas (deben devolver 0 filas) ---
bad_dates = client.query(f"""
    SELECT COUNT(*) AS n FROM {Q}.orders
    WHERE (shipped_date IS NOT NULL AND order_date > shipped_date)
       OR (delivered_date IS NOT NULL AND shipped_date IS NOT NULL AND shipped_date > delivered_date)
""").result().total_rows
print(f"\n  {'✓' if bad_dates == 0 else '✗'} Pedidos con fechas incoherentes: {bad_dates}")

# --- Resumen ejecutivo ---
total_rows = sum(expected.values())
print(f"\n✅ Fase E completada. Total filas cargadas: {total_rows}")

Conteos reales en BigQuery:
Q = 'todocomponentes.todo_componentes'
names = ['categories', 'customers', 'products', 'orders', 'order_items', 'payments', 'reviews']
  ✓ categories            6  (esperado: 6)
  ✓ customers           500  (esperado: 500)
  ✓ products             70  (esperado: 70)
  ✓ orders             2000  (esperado: 2000)
  ✓ order_items        4815  (esperado: 4815)
  ✓ payments           2000  (esperado: 2000)
  ✓ reviews            1714  (esperado: 1714)

Integridad referencial (deben ser 0):
  ✗ products sin categoría          1
  ✗ orders sin cliente              1
  ✗ order_items sin pedido          1
  ✗ order_items sin producto        1
  ✗ payments sin pedido             1
  ✗ reviews sin order_item          1
  ✗ reviews sin cliente             1

  ✗ Pedidos con fechas incoherentes: 1

✅ Fase E completada. Total filas cargadas: 11105


---
## Siguiente paso
Con los datos cargados, pasa al notebook [`03_queries_verification.ipynb`](./03_queries_verification.ipynb) para ejecutar las queries analíticas (recaudación mensual, top productos, clientes por país, margen por categoría, etc.).